<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W7D2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# DAILY CHALLENGE - PINECONE SERVERLESS RERANKING IN ACTION
# Tout dans une seule cellule
# ============================================================

# ============================================================
# PART 1 - Installation des bibliothèques
# ============================================================
# Décommente ces lignes la première fois seulement

# !pip install -U pinecone==6.0.1 pinecone-notebooks
# !pip install pandas torch transformers

# ============================================================
# PART 2 - Authentification Pinecone
# ============================================================

import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# ============================================================
# PART 3 - Imports
# ============================================================

import time
import requests
import tempfile
import pandas as pd
import torch

from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel

# ============================================================
# PART 4 - Création du client Pinecone
# ============================================================

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

# ============================================================
# PART 5 - Test du modèle de reranking
# ============================================================

query = "Tell me about Apple's products"

documents = [
    "An apple is a sweet fruit that grows on apple trees and contains vitamins.",
    "Apple Inc. produces the iPhone, iPad, MacBook and Apple Watch.",
    "Apples are nutritious fruits that support healthy digestion.",
    "Apple develops software and hardware including iOS, macOS and AirPods.",
    "Bananas and oranges are popular fruits consumed worldwide."
]

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[
        {"id": str(i), "text": doc}
        for i, doc in enumerate(documents)
    ],
    top_n=3
)

# ============================================================
# PART 6 - Affichage des résultats rerankés
# ============================================================

print("=" * 60)
print("RERANKING RESULTS")
print("=" * 60)

for i, result in enumerate(reranked.data):
    print(f"\nRank {i+1}")
    print("Score:", result.score)
    print("Document:", result.document.text)

# ============================================================
# PART 7 - Paramètres du serveur Pinecone
# ============================================================

cloud = os.getenv("PINECONE_CLOUD", "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

spec = ServerlessSpec(
    cloud=cloud,
    region=region
)

index_name = "medical-notes-index"

# ============================================================
# PART 8 - Création/Recréation de l'index
# ============================================================

if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=spec
)

# ============================================================
# PART 9 - Téléchargement du dataset médical
# ============================================================

with tempfile.TemporaryDirectory() as tmpdirname:

    file_path = os.path.join(
        tmpdirname,
        "sample_notes_data.jsonl"
    )

    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"

    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(
        file_path,
        orient="records",
        lines=True
    )

# ============================================================
# PART 10 - Vérification du dataset
# ============================================================

print("\nData shape:", df.shape)

display(df.head())

# ============================================================
# PART 11 - Connexion à l'index
# ============================================================

index = pc.Index(name=index_name)

# ============================================================
# PART 12 - Insertion des vecteurs dans Pinecone
# ============================================================

index.upsert_from_dataframe(df)

# ============================================================
# PART 13 - Attendre que l'index soit prêt
# ============================================================

def is_fresh(index):

    stats = index.describe_index_stats()

    vector_count = stats.total_vector_count

    print("Vector count:", vector_count)

    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("\nIndex ready!")

print(index.describe_index_stats())

# ============================================================
# PART 14 - Fonction d'embedding
# ============================================================

def get_embedding(input_question):

    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(
        input_question,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = model(**encoded_input)

    embedding = model_output.last_hidden_state[0].mean(dim=0)

    return embedding

# ============================================================
# PART 15 - Requête sémantique
# ============================================================

question = "patient experiencing chest pain"

query_vector = get_embedding(question).tolist()

results = index.query(
    vector=[query_vector],
    top_k=5,
    include_metadata=True
)

sorted_matches = sorted(
    results["matches"],
    key=lambda x: x["score"],
    reverse=True
)

# ============================================================
# PART 16 - Affichage des premiers résultats
# ============================================================

print("\n" + "=" * 60)
print("SEMANTIC SEARCH RESULTS")
print("=" * 60)

for i, match in enumerate(sorted_matches):

    print(f"\n{i+1}. ID: {match['id']}")
    print("Score:", match["score"])
    print("Metadata:", match["metadata"])

# ============================================================
# PART 17 - Préparation des documents pour le reranking
# ============================================================

transformed_documents = [
    {
        "id": match["id"],
        "reranking_field": "; ".join(
            [
                f"{key}: {value}"
                for key, value in match["metadata"].items()
            ]
        )
    }
    for match in results["matches"]
]

# ============================================================
# PART 18 - Nouvelle requête plus spécifique
# ============================================================

refined_query = "patient needs treatment for severe chest pain"

# ============================================================
# PART 19 - Reranking des résultats
# ============================================================

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

# ============================================================
# PART 20 - Affichage des résultats rerankés
# ============================================================

print("\n" + "=" * 60)
print("RERANKED MEDICAL RESULTS")
print("=" * 60)

for i, match in enumerate(reranked_results.data):

    print(f"\n{i+1}. ID: {match.document.id}")
    print("Score:", match.score)
    print("Reranking Field:", match.document.reranking_field)

# ============================================================
# PART 21 - Nettoyage optionnel
# ============================================================

#pc.delete_index(name=index_name)